# 84장 · 요구사항 변경·코드 검토

팀에서 기존 투두에 마감일과 우선순위를 추가해 달라고 했습니다. 예전 할 일도 계속 읽을 수 있어야 하고 제목만 고쳤을 때 마감일이 지워져서는 안 됩니다. 실제 SQL로 기존 행을 이관하고 두 열을 함께 수정해 봅니다.

실행을 확인한 커널은 Python 3.12.15입니다.

Python 3 커널을 선택하고 위에서부터 실행하세요. 과제 셀에 직접 코드를 작성한 뒤 바로 아래에서 풀이를 펼쳐 보세요.

## 기존 할 일에 새 필드 추가하기

새로 만들 때와 예전 행을 읽을 때 모두 due_date는 비어 있을 수 있고 priority의 기본값은 normal입니다. PostgreSQL에서는 날짜 타입을 사용합니다. 여기서는 SQLite의 ISO 날짜 문자열과 CHECK 조건으로 같은 동작을 좁혀 봅니다.



**볼 곳**　‘기존 할 일’를 ‘새 열 기본값’에 남기는 시점과 ‘이전 행 보존’를 확인하는 시점을 구분하세요.

**애니메이션의 의미**　새 열을 추가해도 이전 할 일의 제목은 유지됩니다.


### 셀 01 · 기존 행이 있는 테이블에 열 추가하기

변경 전의 복습 한 건을 넣고 ADD COLUMN을 적용합니다.

In [1]:
import sqlite3
db = sqlite3.connect(":memory:")
db.execute("CREATE TABLE todos(id INTEGER PRIMARY KEY, title TEXT NOT NULL, completed INTEGER NOT NULL)")
db.execute("INSERT INTO todos VALUES(1, '복습', 0)")
db.execute("ALTER TABLE todos ADD COLUMN due_date TEXT")
db.execute("ALTER TABLE todos ADD COLUMN priority TEXT NOT NULL DEFAULT 'normal' CHECK(priority IN ('low','normal','high'))")
print(db.execute("SELECT title, due_date, priority FROM todos WHERE id=1").fetchone())

('복습', None, 'normal')


기존 제목은 복습으로 유지되고 새 필드는 None과 normal입니다. 예전 행에 존재하지 않았던 마감일을 임의의 날짜로 채우지 않았습니다.

## 고칠 열과 값을 정해 두기

마감일과 우선순위가 함께 온 경우에는 두 열만 고치는 고정 SQL을 선택할 수 있습니다. 입력에서 직접 열 이름을 만들어 붙이지 않습니다. 한 필드가 누락된 요청과 일부러 null을 보내 날짜를 지우는 요청도 구분합니다.



**볼 곳**　‘변경 요청’가 가운데 ‘허용 열 확인’에 도착한 뒤 오른쪽에 어떤 결과가 남는지 보세요.

**애니메이션의 의미**　허용한 열에만 새 값을 적용합니다.


### 셀 02 · 두 열의 고정 UPDATE와 바인딩 값

날짜 문자열은 실제 달력 날짜인지 확인한 뒤 SQL 자리표시자에 넣습니다.

날짜와 우선순위를 저장한 다음, 같은 행의 제목만 바꿔 보세요. 제목 변경 뒤에도 날짜는 2027-04-02, 우선순위는 high여야 합니다. 다음 과제에서 날짜를 직접 비울 때와 비교합니다.

In [2]:
from datetime import date
due = date.fromisoformat("2027-04-02").isoformat()
with db:
    db.execute("UPDATE todos SET due_date=?, priority=? WHERE id=?", (due, "high", 1))
print(db.execute("SELECT title, due_date, priority FROM todos WHERE id=1").fetchone())

('복습', '2027-04-02', 'high')


복습·2027-04-02·high가 나옵니다. 바인딩 값은 SQL에 적힌 열 순서와 같습니다. 전체 T1 구현은 허용하는 변경 필드 조합별로 문장을 고릅니다.

### 제목만 바꾼 뒤 기존 필드 확인하기

아래 셀을 실행한 뒤 바로 앞의 결과와 비교하세요.

In [3]:
with db:
    db.execute("UPDATE todos SET title=? WHERE id=?", ("복습 완료 예정", 1))
print(db.execute("SELECT title, due_date, priority FROM todos WHERE id=1").fetchone())

('복습 완료 예정', '2027-04-02', 'high')


UPDATE에 title만 적었습니다. 날짜와 우선순위를 다시 읽어 두 값이 유지되는지 대조하세요.

### 과제 03 · 우선순위는 남기고 마감일만 지우기

기존 마감일을 NULL로 변경하고 due_date와 priority를 다시 읽으세요.

In [4]:
# 1번의 due_date만 None으로 설정하세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>with db:
    db.execute(&#x27;UPDATE todos SET due_date=? WHERE id=?&#x27;, (None, 1))
print(db.execute(&#x27;SELECT due_date, priority FROM todos WHERE id=?&#x27;, (1,)).fetchone())</code></pre>
<p>실행 결과</p><pre><code>(None, &#x27;high&#x27;)</code></pre>
<p>(None, &#x27;high&#x27;)입니다. due_date를 생략하는 요청에서는 열을 건드리지 않고, 명시적으로 null을 보낼 때만 비웁니다.</p>
</details>

### 셀 04 · 허용하지 않는 우선순위 거절

데이터 규칙이 저장 시점에도 적용되는지 확인하세요.

In [5]:
try:
    with db:
        db.execute("UPDATE todos SET priority=? WHERE id=?", ("urgent", 1))
except sqlite3.IntegrityError:
    print("허용하지 않는 우선순위")
print(db.execute("SELECT priority FROM todos WHERE id=1").fetchone()[0])

허용하지 않는 우선순위
high


오류 뒤에도 high가 남습니다. 코드 검토에서는 기본값뿐 아니라 잘못된 입력을 모두 검사한 뒤 한 번에 반영하는지 살펴보세요.

## 돌아보기

검토할 diff에서 기존 ID·제목·완료 필드를 바꾸는 줄이 있는지, 이전 데이터가 마감일 없이 읽히는지, 빈 날짜와 필드 누락을 구별하는지 찾으세요.